# 🚀 ColabDrive Downloader & Merger
Công cụ tải video từ mọi nguồn (YouTube, Facebook, TikTok, Torrent/Magnet, Direct Link, luồng trực tiếp m3u8/mpd) và ghép video tốc độ cao trực tiếp vào **Google Drive**.

**Đặc điểm nổi bật:**
- Tự động vượt tường lửa chặn tải (HTTP 403 Forbidden) của Seenow/VTVcab.
- Hỗ trợ tải hàng loạt link cùng lúc, tự chọn chế độ tải riêng biệt hoặc ghép nối tự động thành 1 video duy nhất.
- Lưu trữ trực tiếp vào Google Drive để không bị mất dữ liệu.
- Ẩn toàn bộ mã nguồn phức tạp dưới dạng giao diện trực quan.

**Hướng dẫn:** Chạy từng ô từ trên xuống dưới bằng nút ▶️ bên trái.


In [ ]:
#@title 📦 Bước 1: Cài đặt công cụ và thư viện nền tảng (Chạy 1 lần)
#@markdown Chạy ô này để tải và cài đặt các thư viện cần thiết (`yt-dlp`, `ffmpeg`, `aria2`).

import os
import sys

print("⏳ Đang cài đặt/cập nhật thư viện yt-dlp...")
os.system("pip install --force-reinstall -q yt-dlp")

print("⏳ Đang cài đặt aria2c và ffmpeg...")
os.system("apt-get update -y -q && apt-get install -y -q ffmpeg aria2")

print("✅ Đã cài đặt hoàn tất các công cụ!")


In [ ]:
#@title 💾 Bước 2: Kết nối Google Drive
#@markdown Chạy ô này để liên kết với tài khoản Google Drive của bạn. File tải về sẽ được lưu trữ trực tiếp trên Drive.

from google.colab import drive
import os

print("⏳ Vui lòng nhấn vào liên kết xuất hiện để cấp quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("✅ Đã kết nối Google Drive thành công!")
print("📁 Thư mục gốc Drive của bạn: /content/drive/MyDrive/")


In [ ]:
#@title ⬇️ Bước 3: Tải & Ghép Video (Mọi Link / Mọi Nguồn)
#@markdown - **Nhập link video:** Dán 1 hoặc nhiều link chính (.mpd, .m3u8, Youtube...). Nếu nhiều link, ngăn cách chúng bằng **dấu phẩy `,`** hoặc **dấu xuống dòng**.

LINK_VIDEO = "" #@param {type:"string"}
CHE_DO_TAI = "T\u1ea3i v\u00e0 T\u1ef1 \u0111\u1ed9ng gh\u00e9p th\u00e0nh 1 file" #@param ["T\u1ea3i ri\u00eang bi\u1ec7t t\u1eebng file", "T\u1ea3i v\u00e0 T\u1ef1 \u0111\u1ed9ng gh\u00e9p th\u00e0nh 1 file"]
THU_MUC_DRIVE = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
TEN_FILE_LUU = "video.mp4" #@param {type:"string"}
TAI_NHANH_ARIA2 = True #@param {type:"boolean"}

import os
import re
import glob
import shutil
import yt_dlp

if not LINK_VIDEO.strip():
    raise ValueError("❌ Bạn chưa nhập link video!")

temp_dir = "/content/temp_download"
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir, exist_ok=True)
os.makedirs(THU_MUC_DRIVE, exist_ok=True)

# Tách danh sách link
raw_input = LINK_VIDEO.strip()
raw_input = raw_input.replace(',', '\n').replace(';', '\n')
input_lines = [line.strip() for line in raw_input.split('\n') if line.strip()]

print(f"🎬 Bắt đầu xử lý {len(input_lines)} link video đầu vào...")
downloaded_files = []

for idx, url in enumerate(input_lines):
    print(f"\n🚀 [{idx+1}/{len(input_lines)}] Đang tải: {url}")
    
    # Thư mục chứa tạm thời cho từng file
    sub_temp = os.path.join(temp_dir, f"part_{idx:03d}")
    os.makedirs(sub_temp, exist_ok=True)
    
    is_torrent = url.startswith('magnet:?') or url.endswith('.torrent') or '.torrent?' in url
    if is_torrent:
        # Tải Torrent
        cmd = f'aria2c --seed-time=0 --summary-interval=5 -d "{sub_temp}" "{url}"'
        os.system(cmd)
        video_extensions = ('.mp4', '.mkv', '.avi', '.ts', '.mov', '.flv', '.webm', '.m4v')
        found_videos = []
        for root, dirs, files in os.walk(sub_temp):
            for f in files:
                if f.lower().endswith(video_extensions):
                    found_videos.append(os.path.join(root, f))
        if found_videos:
            found_videos.sort(key=os.path.getsize, reverse=True)
            downloaded_files.append(found_videos[0])
        else:
            print(f"❌ Lỗi: Không tìm thấy file video nào trong torrent {idx+1}!")
    else:
        # Tải URL thường với cấu hình vượt tường lửa (Referer & User-Agent)
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Referer': 'https://vtvprime.vn/',
            'Origin': 'https://vtvprime.vn'
        }
        
        ydl_opts = {
            'outtmpl': os.path.join(sub_temp, 'video.%(ext)s'),
            'format': 'bestvideo+bestaudio/best',
            'merge_output_format': 'mp4',
            'quiet': False,
            'no_warnings': False,
            'http_headers': headers
        }
        
        if TAI_NHANH_ARIA2:
            ydl_opts['external_downloader'] = 'aria2c'
            ydl_opts['external_downloader_args'] = [
                '-x16', '-s16', '-k1M',
                '--header=Referer: https://vtvprime.vn/',
                '--header=Origin: https://vtvprime.vn'
            ]
            
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
            parts = [f for f in glob.glob(os.path.join(sub_temp, '*')) if not f.endswith('.part') and not f.endswith('.ytdl')]
            if parts:
                downloaded_files.append(parts[0])
                print(f"✅ Tải thành công phần {idx+1}!")
            else:
                raise Exception("Không tìm thấy file tải về.")
        except Exception as e:
            print(f"⚠️ Gặp lỗi tải ({e}). Thử chế độ tương thích không ghép kênh...")
            ydl_opts['format'] = 'best'
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([url])
                parts = [f for f in glob.glob(os.path.join(sub_temp, '*')) if not f.endswith('.part') and not f.endswith('.ytdl')]
                if parts:
                    downloaded_files.append(parts[0])
                    print(f"✅ Tải tương thích thành công phần {idx+1}!")
                else:
                    print(f"❌ Thất bại hoàn toàn phần {idx+1}!")
            except Exception as e2:
                print(f"❌ Lỗi tải phần {idx+1}: {e2}")

if not downloaded_files:
    raise RuntimeError("❌ Không có video nào được tải về thành công!")

# 3. Xử lý lưu trữ theo chế độ chọn
if CHE_DO_TAI == "Tải riêng biệt từng file":
    print("\n📂 Chế độ: Lưu riêng biệt các file...")
    for i, fp in enumerate(downloaded_files):
        ext = os.path.splitext(fp)[1]
        name_parts = os.path.splitext(TEN_FILE_LUU)
        suffix = f"_{i+1}" if len(downloaded_files) > 1 else ""
        final_name = f"{name_parts[0]}{suffix}{ext}"
        dest = os.path.join(THU_MUC_DRIVE, final_name)
        shutil.move(fp, dest)
        print(f"💾 Đã lưu: {dest}")
    print("🎉 ĐÃ HOÀN THÀNH TẢI RIÊNG BIỆT!")
    
else:
    print("\n🔄 Chế độ: Tự động ghép các file đã tải thành 1 video duy nhất...")
    
    if len(downloaded_files) == 1:
        # Chỉ có 1 file thì move thẳng
        fp = downloaded_files[0]
        ext = os.path.splitext(fp)[1]
        final_name = os.path.splitext(TEN_FILE_LUU)[0] + ext
        dest = os.path.join(THU_MUC_DRIVE, final_name)
        shutil.move(fp, dest)
        print(f"💾 Chỉ có 1 file, đã di chuyển trực tiếp lên Drive: {dest}")
    else:
        # Tạo filelist để ghép nối bằng FFmpeg concat
        filelist_txt = os.path.join(temp_dir, "filelist.txt")
        with open(filelist_txt, 'w', encoding='utf-8') as fl:
            for fp in downloaded_files:
                fl.write(f"file '{fp}'\n")
                
        local_merged = os.path.join(temp_dir, "final_output.mp4")
        print("⚡ Đang ghép nối các phần video cực nhanh (Không encode lại)... ")
        cmd = f'ffmpeg -f concat -safe 0 -i "{filelist_txt}" -c copy -y "{local_merged}" -v warning'
        status = os.system(cmd)
        
        if status == 0 and os.path.exists(local_merged) and os.path.getsize(local_merged) > 0:
            final_name = os.path.splitext(TEN_FILE_LUU)[0] + ".mp4"
            dest = os.path.join(THU_MUC_DRIVE, final_name)
            shutil.move(local_merged, dest)
            print(f"🎉 GHÉP NỐI THÀNH CÔNG! Đã lưu file gộp tại: {dest}")
        else:
            print("❌ Lỗi ghép nối bằng FFmpeg! Tiến hành sao lưu các phần riêng lẻ sang Drive làm dự phòng...")
            for i, fp in enumerate(downloaded_files):
                ext = os.path.splitext(fp)[1]
                dest = os.path.join(THU_MUC_DRIVE, f"{os.path.splitext(TEN_FILE_LUU)[0]}_part_{i+1}{ext}")
                shutil.move(fp, dest)
                print(f"💾 Đã lưu dự phòng: {dest}")

try:
    shutil.rmtree(temp_dir)
except: pass


In [ ]:
#@title 🎬 Bước 4: Ghép nhiều video có sẵn trên Drive / Colab
#@markdown Có 2 chế độ ghép:
#@markdown 1. **Ghép từ thư mục:** Ghép tất cả file video trong thư mục chỉ định (sắp xếp theo tên hoặc ngày tạo).
#@markdown 2. **Ghép theo danh sách:** Chỉ định chính xác các file cần ghép theo thứ tự mong muốn.

CHE_DO_GHEP = "Gh\u00e9p t\u1eeb th\u01b0 m\u1ee5c" #@param ["Gh\u00e9p t\u1eeb th\u01b0 m\u1ee5c", "Gh\u00e9p theo danh sách file tự chọn"]
DUONG_DAN_DAU_VAO = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
THU_MUC_DRIVE = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
TEN_FILE_LUU = "final_merged_video.mp4" #@param {type:"string"}
SAP_XEP_THEO = "T\u00ean A-Z" #@param ["T\u00ean A-Z", "Th\u1eddi gian tạo (cũ -> mới)", "Th\u1eddi gian tạo (mới -> cũ)"]
KIEU_GHEP = "Gh\u00e9p si\u00eau tốc (Không encode l\u1ea1i)" #@param ["Gh\u00e9p si\u00eau tốc (Không encode l\u1ea1i)", "Gh\u00e9p chất lượng (Có encode l\u1ea1i - tương th\u00edch cao)"]

import os
import glob
import shutil

# 1. Thu thập danh sách file
video_extensions = ('.mp4', '.mkv', '.avi', '.ts', '.mov', '.flv', '.webm', '.m4v')
files_to_merge = []

if CHE_DO_GHEP == "Ghép từ thư mục":
    if not os.path.exists(DUONG_DAN_DAU_VAO):
        raise FileNotFoundError(f"❌ Không tìm thấy thư mục đầu vào: {DUONG_DAN_DAU_VAO}")
    
    all_files = []
    for f in os.listdir(DUONG_DAN_DAU_VAO):
        full_path = os.path.join(DUONG_DAN_DAU_VAO, f)
        if os.path.isfile(full_path) and f.lower().endswith(video_extensions):
            all_files.append(full_path)
            
    if not all_files:
        raise ValueError(f"❌ Không tìm thấy video nào trong thư mục: {DUONG_DAN_DAU_VAO}")
        
    if SAP_XEP_THEO == "Tên A-Z":
        all_files.sort()
    elif SAP_XEP_THEO == "Thời gian tạo (cũ -> mới)":
        all_files.sort(key=os.path.getctime)
    elif SAP_XEP_THEO == "Thời gian tạo (mới -> cũ)":
        all_files.sort(key=os.path.getctime, reverse=True)
        
    files_to_merge = all_files
else:
    lines = [line.strip() for line in DUONG_DAN_DAU_VAO.strip().split('\n') if line.strip()]
    for line in lines:
        if os.path.exists(line):
            files_to_merge.append(line)
        else:
            print(f"⚠️ Cảnh báo: File không tồn tại và sẽ bị bỏ qua: {line}")
            
    if not files_to_merge:
        raise ValueError("❌ Không có file hợp lệ nào để tiến hành ghép!")

print(f"🎥 Tổng số file chuẩn bị ghép: {len(files_to_merge)}")
for i, f in enumerate(files_to_merge):
    print(f"  {i+1}. {os.path.basename(f)}")

# Tạo file danh sách cho FFmpeg
filelist_path = "/content/filelist.txt"
with open(filelist_path, 'w', encoding='utf-8') as f:
    for fp in files_to_merge:
        escaped_fp = fp.replace("'", "'\\''")
        f.write(f"file '{escaped_fp}'\n")

local_output_path = os.path.join("/content", TEN_FILE_LUU)
if os.path.exists(local_output_path):
    os.remove(local_output_path)

os.makedirs(THU_MUC_DRIVE, exist_ok=True)

# 2. Thực hiện ghép nối bằng FFmpeg
if KIEU_GHEP == "Ghép siêu tốc (Không encode lại)":
    print("\n⚡ Đang thực hiện ghép siêu tốc (chỉ copy stream, không encode lại)...")
    cmd = f'ffmpeg -f concat -safe 0 -i "{filelist_path}" -c copy -y "{local_output_path}" -v warning'
else:
    print("\n🎬 Đang thực hiện ghép có encode lại (sẽ chuyển đổi về H.264 + AAC)...")
    cmd = f'ffmpeg -f concat -safe 0 -i "{filelist_path}" -vf "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2" -c:v libx264 -preset fast -crf 23 -c:a aac -b:a 192k -pix_fmt yuv420p -y "{local_output_path}" -v warning'

status = os.system(cmd)

if status == 0 and os.path.exists(local_output_path) and os.path.getsize(local_output_path) > 0:
    dest_drive_path = os.path.join(THU_MUC_DRIVE, TEN_FILE_LUU)
    print(f"💾 Đang di chuyển file đã ghép lên Google Drive: {dest_drive_path}...")
    shutil.move(local_output_path, dest_drive_path)
    print("🎉 HOÀN THÀNH GHÉP VIDEO!")
    print(f"📁 File đã lưu tại: {dest_drive_path}")
else:
    print("❌ Lỗi xảy ra trong quá trình ghép video bằng FFmpeg.")
    if os.path.exists(local_output_path):
        try: os.remove(local_output_path)
        except: pass

if os.path.exists(filelist_path):
    os.remove(filelist_path)
